# Hidden Markov Model (HMM) - Dishonest Casino Example

**Depth Level:** Medium 📘

This notebook introduces Hidden Markov Models through the classic "Dishonest Casino" example.

## What You'll Learn
1. **Markov Chains** - Basics, transition matrices, stable states
2. **HMM Structure** - Initial distribution (π), Transition matrix (A), Emission matrix (B)
3. **Forward Algorithm** - Computing likelihood of observations
4. **Viterbi Algorithm** - Finding the most likely hidden state sequence
5. **Baum-Welch Algorithm** - Learning HMM parameters from data

## The Dishonest Casino Problem

A casino dealer switches between a **fair die** and a **loaded die** to cheat. The loaded die is biased towards rolling 6.

- **Hidden States**: Which die is being used (Fair or Loaded)
- **Observations**: The dice roll values (1, 2, 3, 4, 5, 6)

**Goal**: Given a sequence of dice rolls, can we infer when the dealer switched dice?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hmmlearn import hmm

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

---
## Part 1: Markov Chains Basics

### What is a Markov Process?

A **Markov process** (or Markov chain in discrete space) is a stochastic process where:
> The future state depends **only** on the current state, not on the history.

$$P(X_{t+1} | X_t, X_{t-1}, ..., X_1) = P(X_{t+1} | X_t)$$

### Transition Matrix

The probability of transitioning between states is captured in a **transition matrix**.

In [ ]:
# Transition Matrix for Dishonest Casino
# States: 0 = Fair Die, 1 = Loaded Die

A = np.array([
    [0.95, 0.05],  # From Fair: stay Fair, switch to Loaded
    [0.10, 0.90]   # From Loaded: switch to Fair, stay Loaded
])

print("Transition Matrix A:")
print(A)
print(f"\nRows sum to 1: {A.sum(axis=1)}")

### Finding the Stable State (Eigenvalue Method)

The **stable state** is the eigenvector corresponding to eigenvalue = 1.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(A.T)
print(f"Eigenvalues: {eigenvalues}")

idx = np.argmin(np.abs(eigenvalues - 1))
stable_state = eigenvectors[:, idx].real
stable_state = stable_state / stable_state.sum()

print(f"\nStable state distribution:")
print(f"  P(Fair die) = {stable_state[0]:.4f}")
print(f"  P(Loaded die) = {stable_state[1]:.4f}")

---
## Part 2: Hidden Markov Model Structure

### Three Components:
1. **π (pi)** - Initial state distribution
2. **A** - Transition matrix
3. **B** - Emission matrix

In [ ]:
# Define HMM parameters
pi = np.array([0.5, 0.5])

A = np.array([
    [0.95, 0.05],
    [0.10, 0.90]
])

# Emission: P(observation | state)
B = np.array([
    [1/6, 1/6, 1/6, 1/6, 1/6, 1/6],  # Fair die: uniform
    [0.1, 0.1, 0.1, 0.1, 0.1, 0.5]   # Loaded die: biased to 6
])

print("B (emission matrix):")
print(f"       Roll 1   Roll 2   Roll 3   Roll 4   Roll 5   Roll 6")
print(f"Fair:   {B[0]}")
print(f"Loaded: {B[1]}")

In [ ]:
def generate_hmm_sequence(pi, A, B, n_steps):
    n_states, n_obs = B.shape
    states, observations = [], []
    current_state = np.random.choice(n_states, p=pi)
    
    for _ in range(n_steps):
        states.append(current_state)
        obs = np.random.choice(n_obs, p=B[current_state])
        observations.append(obs)
        current_state = np.random.choice(n_states, p=A[current_state])
    
    return np.array(states), np.array(observations)

true_states, observations = generate_hmm_sequence(pi, A, B, n_steps=50)
print(f"Observations (first 20): {observations[:20] + 1}")
print(f"True states (first 20):  {true_states[:20]}")

---
## Part 3: Forward Algorithm

$$\alpha_t(j) = \left[ \sum_i \alpha_{t-1}(i) \cdot a_{ij} \right] \cdot b_j(o_t)$$

In [ ]:
def forward_algorithm(observations, pi, A, B):
    T, N = len(observations), len(pi)
    alpha = np.zeros((T, N))
    
    alpha[0] = pi * B[:, observations[0]]
    
    for t in range(1, T):
        for j in range(N):
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, observations[t]]
    
    return alpha, np.sum(alpha[-1])

alpha, likelihood = forward_algorithm(observations, pi, A, B)
print(f"Likelihood: {likelihood:.2e}")
print(f"Log-likelihood: {np.log(likelihood):.4f}")

---
## Part 4: Viterbi Algorithm

$$v_t(j) = \max_i \left[ v_{t-1}(i) \cdot a_{ij} \right] \cdot b_j(o_t)$$

In [ ]:
def viterbi_algorithm(observations, pi, A, B):
    T, N = len(observations), len(pi)
    v = np.zeros((T, N))
    backpointers = np.zeros((T, N), dtype=int)
    
    v[0] = pi * B[:, observations[0]]
    
    for t in range(1, T):
        for j in range(N):
            probs = v[t-1] * A[:, j]
            best_prev = np.argmax(probs)
            v[t, j] = probs[best_prev] * B[j, observations[t]]
            backpointers[t, j] = best_prev
    
    best_path = [np.argmax(v[-1])]
    for t in range(T-1, 0, -1):
        best_path.append(backpointers[t, best_path[-1]])
    
    return np.array(best_path[::-1]), v[-1, best_path[0]]

predicted_states, best_prob = viterbi_algorithm(observations, pi, A, B)
accuracy = (predicted_states == true_states).mean()
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].bar(range(len(observations)), observations + 1, color='steelblue', alpha=0.7)
axes[0].set_ylabel('Dice Roll')
axes[0].set_title('Observed Dice Rolls')

axes[1].step(range(len(true_states)), true_states, where='mid', color='green', linewidth=2)
axes[1].set_ylabel('State')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Fair', 'Loaded'])
axes[1].set_title('True Hidden States')
axes[1].fill_between(range(len(true_states)), true_states, step='mid', alpha=0.3, color='green')

axes[2].step(range(len(predicted_states)), predicted_states, where='mid', color='red', linewidth=2)
axes[2].set_ylabel('State')
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(['Fair', 'Loaded'])
axes[2].set_title(f'Predicted States (Viterbi) - Accuracy: {accuracy:.1%}')
axes[2].fill_between(range(len(predicted_states)), predicted_states, step='mid', alpha=0.3, color='red')

plt.tight_layout()
plt.show()

---
## Part 5: Verification with hmmlearn

In [ ]:
model = hmm.CategoricalHMM(n_components=2, n_iter=100, random_state=42)
model.startprob_ = pi
model.transmat_ = A
model.emissionprob_ = B

obs_2d = observations.reshape(-1, 1)

log_prob_lib = model.score(obs_2d)
_, states_lib = model.decode(obs_2d)

print(f"Log-likelihood (hmmlearn): {log_prob_lib:.4f}")
print(f"Log-likelihood (our impl): {np.log(likelihood):.4f}")
print(f"Viterbi match: {(states_lib == predicted_states).all()}")

---
## Summary

| Algorithm | Purpose | Complexity |
|-----------|---------|------------|
| Forward | P(observations) | O(N²T) |
| Viterbi | Best state sequence | O(N²T) |
| Baum-Welch | Learn parameters | O(N²T) per iter |